# 115 — Diagnose checkpoint 109 and compare a small scalar Q

Run alongside the two new source-data workers. This notebook uses the **existing immutable 800-root fresh8 snapshot** and the same 640/160 root split as notebook 109. The checkpoint analysis reports oracle headroom, root-weighted pair accuracy, score spread, and exploratory Q-margin gates. Optional controls train a compact sigmoid Q10 and a state-only version on the same root-MC labels. Neither a gate threshold nor a checkpoint chosen on these already-inspected validation roots is a confirmatory result. No simulator run occurs here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXTRAS = 'sim'
SETUP_ENV = False
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
import torch
from pnp.smolvla_small_q import (
    load_fresh8_roots, analyze_existing_checkpoint,
    train_small_q, SmallTrainConfig)

SNAPSHOT_DIGEST = '23241dbe9dfeaec615e1f52d'
LARGE_CHECKPOINT = Path('/content/drive/MyDrive/pnp_smolvla_q10_fresh8') / SNAPSHOT_DIGEST / 'root_mc/checkpoint_step_002000.pt'
CACHE_ROOT = Path('/content/smolvla_q10_fresh8_cache')
OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_small_q_fresh8') / SNAPSHOT_DIGEST
RUN_SMALL_Q = False
RUN_STATE_ONLY = False
assert LARGE_CHECKPOINT.is_file(), 'Mount Drive containing notebook 109 checkpoint.'
assert torch.cuda.is_available(), 'Select a GPU runtime for the model comparisons.'
snapshot, cache, train, validation = load_fresh8_roots(cache_root=CACHE_ROOT)
assert snapshot['snapshot_digest'] == SNAPSHOT_DIGEST
assert len(train) == 640 and len(validation) == 160
print({'snapshot': SNAPSHOT_DIGEST, 'train_roots': len(train),
       'validation_roots': len(validation), 'gpu': torch.cuda.get_device_name(0)})


In [ ]:
large_report = analyze_existing_checkpoint(
    checkpoint=LARGE_CHECKPOINT, validation=validation,
    snapshot=snapshot, device='cuda')
for key, value in large_report.items():
    print(key, value)


In [ ]:
if RUN_SMALL_Q:
    small_report = train_small_q(
        train=train, validation=validation,
        snapshot_digest=SNAPSHOT_DIGEST,
        output_path=OUTPUT_ROOT/'small_action_q.pt',
        state_only=False, device='cuda',
        config=SmallTrainConfig(updates=2000))
    print(small_report)
else:
    print('Small action-conditioned Q not started; set RUN_SMALL_Q=True.')


In [ ]:
if RUN_STATE_ONLY:
    state_report = train_small_q(
        train=train, validation=validation,
        snapshot_digest=SNAPSHOT_DIGEST,
        output_path=OUTPUT_ROOT/'small_state_only.pt',
        state_only=True, device='cuda',
        config=SmallTrainConfig(updates=2000))
    print(state_report)
else:
    print('State-only control not started; set RUN_STATE_ONLY=True.')
